# Hand YOLO Video Demonstration

Render hand YOLO detections back onto a world-camera video for quick visual inspection.

In [ ]:
from pathlib import Path
import csv
from collections import Counter, defaultdict

import cv2
from IPython.display import Video, display

PROJECT_ROOT = Path("/home/julius/projects/Identification-of-daily-activities-feature-extraction")
DATASET_ROOT = Path("/home/julius/datasets/seminar_object_detection")
EXPORT_ROOT = DATASET_ROOT / "iAssistADL_dataset_export_2026-05-18"
VIDEO_ROOT = EXPORT_ROOT / "world_camera_videos"
YOLO_HAND_ROOT = DATASET_ROOT / "lower_feature_outcome" / "yolo_hand_outcome_csv"
OUTPUT_ROOT = DATASET_ROOT / "video_demonstration" / "hand_yolo"

SELECTED_FULL_ID = "IA_4_H_06_Brief_schreiben1"
RENDER_FULL_VIDEO = True
WINDOW_SECONDS = 30
MAX_BOXES_PER_FRAME = 20
MIN_CONFIDENCE = 0.0

participant_id = "_".join(SELECTED_FULL_ID.split("_")[:4])
video_path = VIDEO_ROOT / participant_id / f"{SELECTED_FULL_ID}_world_camera.mp4"
detections_path = YOLO_HAND_ROOT / f"{SELECTED_FULL_ID}_world_camera.csv"
duration_label = "full_video" if RENDER_FULL_VIDEO else f"{WINDOW_SECONDS}s"
output_video_path = OUTPUT_ROOT / f"{SELECTED_FULL_ID}_hand_yolo_demo_{duration_label}.mp4"

for path in [video_path, detections_path]:
    if not path.exists():
        raise FileNotFoundError(path)

video_path, detections_path, output_video_path

In [ ]:
def read_detections(path, min_confidence=0.0):
    detections_by_frame = defaultdict(list)
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            confidence = float(row["confidence"])
            if confidence < min_confidence:
                continue
            frame_id = int(float(row["frame_id"]))
            detections_by_frame[frame_id].append(
                {
                    "class_name": row.get("class_name", "hand"),
                    "confidence": confidence,
                    "x1": float(row["x1"]),
                    "y1": float(row["y1"]),
                    "x2": float(row["x2"]),
                    "y2": float(row["y2"]),
                }
            )
    return detections_by_frame


cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

detections_by_frame = read_detections(detections_path, MIN_CONFIDENCE)
window_size = int(round(WINDOW_SECONDS * fps))
detection_counts = Counter({frame_id: len(items) for frame_id, items in detections_by_frame.items()})

if RENDER_FULL_VIDEO:
    start_frame = 0
    end_frame = frame_count
    best_score = sum(detection_counts.values())
else:
    best_start = 0
    best_score = -1
    if detection_counts:
        max_start = max(0, frame_count - window_size)
        candidate_starts = range(0, max_start + 1, max(1, int(fps)))
        for start in candidate_starts:
            end = start + window_size
            score = sum(count for frame_id, count in detection_counts.items() if start <= frame_id < end)
            if score > best_score:
                best_start = start
                best_score = score
    start_frame = best_start
    end_frame = min(frame_count, start_frame + window_size)

{
    "fps": fps,
    "frame_count": frame_count,
    "size": (width, height),
    "detection_frames": len(detections_by_frame),
    "detections_in_selected_window": best_score,
    "start_frame": start_frame,
    "end_frame": end_frame,
}

In [ ]:
def draw_hand_boxes(frame, detections, frame_id):
    color = (40, 220, 255)
    text_color = (10, 10, 10)
    detections = sorted(detections, key=lambda item: item["confidence"], reverse=True)[:MAX_BOXES_PER_FRAME]

    for det in detections:
        x1 = max(0, min(frame.shape[1] - 1, int(round(det["x1"]))))
        y1 = max(0, min(frame.shape[0] - 1, int(round(det["y1"]))))
        x2 = max(0, min(frame.shape[1] - 1, int(round(det["x2"]))))
        y2 = max(0, min(frame.shape[0] - 1, int(round(det["y2"]))))

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
        label = f"{det['class_name']} {det['confidence']:.2f}"
        (label_w, label_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        label_y1 = max(0, y1 - label_h - baseline - 8)
        label_y2 = label_y1 + label_h + baseline + 8
        cv2.rectangle(frame, (x1, label_y1), (min(frame.shape[1] - 1, x1 + label_w + 12), label_y2), color, -1)
        cv2.putText(frame, label, (x1 + 6, label_y2 - baseline - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.7, text_color, 2, cv2.LINE_AA)

    overlay = f"{SELECTED_FULL_ID} | frame {frame_id} | hands {len(detections)}"
    cv2.putText(frame, overlay, (24, 42), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3, cv2.LINE_AA)
    cv2.putText(frame, overlay, (24, 42), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (20, 20, 20), 1, cv2.LINE_AA)
    return frame


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
cap = cv2.VideoCapture(str(video_path))
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))
if not writer.isOpened():
    raise RuntimeError(f"Could not create output video: {output_video_path}")

frame_id = start_frame
while frame_id < end_frame:
    ok, frame = cap.read()
    if not ok:
        break

    draw_hand_boxes(frame, detections_by_frame.get(frame_id, []), frame_id)
    writer.write(frame)
    frame_id += 1

cap.release()
writer.release()

output_video_path

In [ ]:
display(Video(str(output_video_path), embed=False, html_attributes="controls muted"))
str(output_video_path)

## Per-Video Detection Rates

List each processed video with the percentage of frames containing 0, 1, exactly 2, or at least 2 hand detections.

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import pandas as pd
import matplotlib.pyplot as plt


def participant_from_full_id(full_id):
    return "_".join(full_id.split("_")[:4])


def get_video_info(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return 0, 0.0
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    cap.release()
    return frames, fps


def count_detections_by_frame(path, min_confidence=0.0):
    counts = Counter()
    confidences = []
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            confidence = float(row["confidence"])
            if confidence < min_confidence:
                continue
            frame_id = int(float(row["frame_id"]))
            counts[frame_id] += 1
            confidences.append(confidence)
    return counts, confidences


rate_rows = []
for csv_path in sorted(YOLO_HAND_ROOT.glob("*_world_camera.csv")):
    full_id = csv_path.name.replace("_world_camera.csv", "")
    participant_id = participant_from_full_id(full_id)
    current_video_path = VIDEO_ROOT / participant_id / f"{full_id}_world_camera.mp4"
    total_frames, current_fps = get_video_info(current_video_path)
    counts, confidences = count_detections_by_frame(csv_path, MIN_CONFIDENCE)

    if total_frames <= 0:
        total_frames = max(counts.keys(), default=-1) + 1

    histogram = Counter(counts.get(frame_id, 0) for frame_id in range(total_frames))
    frames_0 = histogram.get(0, 0)
    frames_1 = histogram.get(1, 0)
    frames_2 = histogram.get(2, 0)
    frames_3plus = sum(value for key, value in histogram.items() if key >= 3)
    detections = sum(counts.values())
    mean_confidence = sum(confidences) / len(confidences) if confidences else 0.0

    rate_rows.append(
        {
            "full_id": full_id,
            "participant": participant_id,
            "frames": total_frames,
            "detections": detections,
            "avg_per_frame": detections / total_frames if total_frames else 0.0,
            "zero_hand_rate": frames_0 / total_frames if total_frames else 0.0,
            "one_hand_rate": frames_1 / total_frames if total_frames else 0.0,
            "two_hand_rate": frames_2 / total_frames if total_frames else 0.0,
            "at_least_two_rate": (frames_2 + frames_3plus) / total_frames if total_frames else 0.0,
            "three_plus_rate": frames_3plus / total_frames if total_frames else 0.0,
            "mean_confidence": mean_confidence,
        }
    )

rate_df = pd.DataFrame(rate_rows).sort_values("at_least_two_rate", ascending=False).reset_index(drop=True)
rate_df.insert(0, "rank", rate_df.index + 1)

summary = pd.DataFrame(
    [
        {
            "videos": len(rate_df),
            "frames": int(rate_df["frames"].sum()),
            "detections": int(rate_df["detections"].sum()),
            "avg_per_frame": rate_df["detections"].sum() / rate_df["frames"].sum(),
            "zero_hand_rate": (rate_df["zero_hand_rate"] * rate_df["frames"]).sum() / rate_df["frames"].sum(),
            "one_hand_rate": (rate_df["one_hand_rate"] * rate_df["frames"]).sum() / rate_df["frames"].sum(),
            "two_hand_rate": (rate_df["two_hand_rate"] * rate_df["frames"]).sum() / rate_df["frames"].sum(),
            "at_least_two_rate": (rate_df["at_least_two_rate"] * rate_df["frames"]).sum() / rate_df["frames"].sum(),
            "three_plus_rate": (rate_df["three_plus_rate"] * rate_df["frames"]).sum() / rate_df["frames"].sum(),
            "mean_confidence": (rate_df["mean_confidence"] * rate_df["detections"]).sum() / rate_df["detections"].sum(),
        }
    ]
)

display(
    summary.style.format(
        {
            "avg_per_frame": "{:.2f}",
            "zero_hand_rate": "{:.1%}",
            "one_hand_rate": "{:.1%}",
            "two_hand_rate": "{:.1%}",
            "at_least_two_rate": "{:.1%}",
            "three_plus_rate": "{:.1%}",
            "mean_confidence": "{:.2f}",
        }
    )
)

visible_columns = [
    "rank",
    "full_id",
    "frames",
    "detections",
    "avg_per_frame",
    "zero_hand_rate",
    "one_hand_rate",
    "two_hand_rate",
    "at_least_two_rate",
    "three_plus_rate",
    "mean_confidence",
]

styled_rates = (
    rate_df[visible_columns]
    .style
    .format(
        {
            "avg_per_frame": "{:.2f}",
            "zero_hand_rate": "{:.1%}",
            "one_hand_rate": "{:.1%}",
            "two_hand_rate": "{:.1%}",
            "at_least_two_rate": "{:.1%}",
            "three_plus_rate": "{:.1%}",
            "mean_confidence": "{:.2f}",
        }
    )
    .background_gradient(subset=["at_least_two_rate"], cmap="Greens")
    .background_gradient(subset=["zero_hand_rate", "one_hand_rate"], cmap="Reds")
    .background_gradient(subset=["mean_confidence"], cmap="Blues")
)

display(styled_rates)

plot_df = rate_df.sort_values("at_least_two_rate", ascending=True)
fig_height = max(6, 0.28 * len(plot_df))
fig, ax = plt.subplots(figsize=(12, fig_height))
ax.barh(plot_df["full_id"], plot_df["at_least_two_rate"], color="#2ca25f", label=">=2 hands")
ax.barh(plot_df["full_id"], plot_df["zero_hand_rate"], left=plot_df["at_least_two_rate"], color="#de2d26", alpha=0.55, label="0 hand")
ax.set_xlim(0, 1)
ax.set_xlabel("frame rate")
ax.set_title("Hand YOLO detection coverage by video")
ax.xaxis.set_major_formatter(lambda value, pos: f"{value:.0%}")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()
